In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
# Load the datasets
results_df = pd.read_csv('../data/results.csv')
races_df = pd.read_csv('../data/races.csv')
drivers_df = pd.read_csv('../data/drivers.csv')
constructors_df = pd.read_csv('../data/constructors.csv')
qualifying_df = pd.read_csv('../data/qualifying.csv')

print('Results shape:', results_df.shape)
print('Races shape:', races_df.shape)
print('Drivers shape:', drivers_df.shape)
print('Constructors shape:', constructors_df.shape)
print('Qualifying shape:', qualifying_df.shape)

# Merge datasets for comprehensive analysis
# Merge results with races
df = results_df.merge(races_df, on='race_id', how='left')
# Merge with drivers
df = df.merge(drivers_df, on='driver_id', how='left')
# Merge with constructors
df = df.merge(constructors_df, on='constructor_id', how='left')
# Merge with qualifying (for grid position)
df = df.merge(qualifying_df, on=['race_id', 'driver_id'], how='left')

print('Merged dataset shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

Results shape: (7600, 9)
Races shape: (1149, 7)
Drivers shape: (616, 5)
Constructors shape: (168, 3)
Qualifying shape: (3017, 7)
Merged dataset shape: (7600, 26)
Columns: ['race_id', 'driver_id', 'constructor_id_x', 'grid', 'position_x', 'position_order', 'points', 'laps', 'status', 'season', 'round', 'race_name', 'date', 'time', 'circuit_id', 'givenName', 'familyName', 'nationality_x', 'dob', 'name', 'nationality_y', 'constructor_id_y', 'position_y', 'q1', 'q2', 'q3']


,race_id,driver_id,constructor_id_x,grid,position_x,position_order,points,laps,status,season,...,familyName,nationality_x,dob,name,nationality_y,constructor_id_y,position_y,q1,q2,q3
0,1950_1,farina,alfa,1,1,1,9.0,70,Finished,1950,...,Farina,Italian,1906-10-30,Alfa Romeo,Swiss,NaN,NaN,NaN,NaN,NaN
1,1950_1,fagioli,alfa,2,2,2,6.0,70,Finished,1950,...,Fagioli,Italian,1898-06-09,Alfa Romeo,Swiss,NaN,NaN,NaN,NaN,NaN
2,1950_1,reg_parnell,alfa,4,3,3,4.0,70,Finished,1950,...,Parnell,British,1911-07-02,Alfa Romeo,Swiss,NaN,NaN,NaN,NaN,NaN
3,1950_1,cabantous,lago,6,4,4,3.0,68,+2 Laps,1950,...,Cabantous,French,1904-10-08,Talbot-Lago,French,NaN,NaN,NaN,NaN,NaN
4,1950_1,rosier,lago,9,5,5,2.0,68,+2 Laps,1950,...,Rosier,French,1905-11-05,Talbot-Lago,French,NaN,NaN,NaN,NaN,NaN


In [4]:
# Basic statistics
df.describe()

,grid,position_order,points,laps,season,round,position_y
count,7600.000000,7600.000000,7600.000000,7600.000000,7600.000000,7600.000000,2858.000000
mean,11.353026,11.421579,2.077893,52.897105,1987.500000,2.776447,10.974808
std,6.952794,6.895830,4.278727,35.322680,21.938854,1.333227,6.238190
min,0.000000,1.000000,0.000000,0.000000,1950.000000,1.000000,1.000000
25%,5.000000,6.000000,0.000000,31.000000,1968.750000,2.000000,6.000000
50%,11.000000,11.000000,0.000000,56.000000,1987.500000,3.000000,11.000000
75%,17.000000,17.000000,2.000000,66.000000,2006.250000,4.000000,16.000000
max,33.000000,33.000000,26.000000,200.000000,2025.000000,7.000000,28.000000


In [5]:
# Check for missing values
df.isnull().sum()

race_id                0
driver_id              0
constructor_id_x       0
grid                   0
position_x             0
position_order         0
points                 0
laps                   0
status                 0
season                 0
round                  0
race_name              0
date                   0
time                5500
circuit_id             0
givenName              0
familyName             0
nationality_x          0
dob                    0
name                   0
nationality_y          0
constructor_id_y    4742
position_y          4742
q1                  4779
q2                  6048
q3                  6659
dtype: int64

In [6]:
# Data types
df.dtypes

race_id              object
driver_id            object
constructor_id_x     object
grid                  int64
position_x           object
position_order        int64
points              float64
laps                  int64
status               object
season                int64
round                 int64
race_name            object
date                 object
time                 object
circuit_id           object
givenName            object
familyName           object
nationality_x        object
dob                  object
name                 object
nationality_y        object
constructor_id_y     object
position_y          float64
q1                   object
q2                   object
q3                   object
dtype: object

In [8]:
# Clean data - drop rows with missing position_order
df = df.dropna(subset=['position_order'])
print('After cleaning shape:', df.shape)

After cleaning shape: (7600, 26)


In [22]:
# EDA: Distribution of Final Positions
fig = px.histogram(df, x='position_order', nbins=20, title='Final Position Distribution')
fig.update_layout(showlegend=False)
fig.write_image('../viz/position_distribution.png')
fig.show()

In [21]:
# EDA: Final Position by Constructor (top 10)
top_constructors = df['name'].value_counts().head(10).index
df_top_const = df[df['name'].isin(top_constructors)]
fig = px.box(df_top_const, x='name', y='position_order', title='Final Position by Constructor (Top 10)')
fig.write_image('../viz/position_by_constructor.png')
fig.show()

In [20]:
# EDA: Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()
fig = px.imshow(corr_matrix, text_auto=True, title='Correlation Heatmap')
fig.write_image('../viz/correlation_heatmap.png')
fig.show()

In [19]:
# EDA: Grid Position vs Final Position
fig = px.scatter(df, x='grid', y='position_order', title='Grid Position vs Final Position')
fig.update_layout(xaxis_title='Grid Position', yaxis_title='Final Position')
fig.write_image('../viz/grid_vs_final.png')
fig.show()

In [13]:
# Preprocessing
# Encode categorical variables
le_constructor = LabelEncoder()
df['constructor_encoded'] = le_constructor.fit_transform(df['name'])

le_driver = LabelEncoder()
df['driver_encoded'] = le_driver.fit_transform(df['driver_id'])

# Select features
features = ['grid', 'constructor_encoded', 'driver_encoded', 'laps']
X = df[features]
y = df['position_order']

# Drop rows with NaN in features
X = X.dropna()
y = y.loc[X.index]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [18]:
# Train Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Predictions
y_pred = rf_model.predict(X_test_scaled)

# Evaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R²: {r2:.4f}')

# Actual vs Predicted plot
fig = px.scatter(x=y_test, y=y_pred, title='Actual vs Predicted Final Position')
fig.add_trace(go.Scatter(x=[y_test.min(), y_test.max()], y=[y_test.min(), y_test.max()], mode='lines', name='Perfect Prediction'))
fig.update_layout(xaxis_title='Actual Position', yaxis_title='Predicted Position')
fig.write_image('../viz/actual_vs_predicted.png')
fig.show()

MAE: 3.07
RMSE: 4.14
R²: 0.6353


In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

fig = px.bar(feature_importance, x='importance', y='feature', orientation='h', title='Feature Importance')
fig.write_image('../viz/feature_importance.png')
fig.show()

In [16]:
from sklearn.pipeline import Pipeline

# Define the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Fit the pipeline
pipeline.fit(X_train, y_train)

# Sample
sample_race = pd.DataFrame({
    'grid': [1],  # pole position
    'constructor_encoded': [le_constructor.transform(['Alfa Romeo'])[0]],
    'driver_encoded': [le_driver.transform(['farina'])[0]],
    'laps': [70]
})

# Make prediction
prediction = pipeline.predict(sample_race)

print('Sample F1 Race Prediction:')
print(f'Predicted Final Position: {prediction[0]:.2f}')

# Save the pipeline
import joblib
joblib.dump(pipeline, '../models/f1_pipeline.joblib')
print('Pipeline saved to ../models/f1_pipeline.joblib')

Sample F1 Race Prediction:
Predicted Final Position: 5.06
Pipeline saved to ../models/f1_pipeline.joblib
